<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
StateGraph Basics
</b></font> </br></p>

---

In [1]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M09-StateGraph-Basics"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.15
langchain-chroma                         1.1.0
langchain-classic                        1.0.4
langchain-community                      0.4.1
langchain-core                           1.3.1
langchain-ollama                         1.1.0
langchain-openai                         1.2.0
langchain-text-splitters                 1.1.2
langgraph                                1.1.9
langgraph-checkpoint                     4.0.2
langgraph-prebuilt                       1.0.10
langgraph-sdk                            0.3.13

IP-Adresse: 34.26.107.195
Hostname: 195.107.26.34.bc.googleusercontent.com
Stadt: North Charleston
Region: South Carolina
Land: US
Koordinaten: 32.8546,-79.9748
Provider: AS396982 Google LLC
Postleitzahl: 29415
Zeitzone: America/New_Y

**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema

# 1 | Übersicht
---



*Warum LangGraph?* hat gezeigt, **warum** LangGraph existiert und einen ersten 1-Node-Graphen gebaut.  
Dieses Modul geht tiefer: Wir bauen einen **vollständigen 2-Node-Graphen** mit mehreren State-Feldern,  
verschiedenen Edge-Typen und vollständiger Visualisierung.



**Was behandelt dieses Modul**

| Thema | Inhalt |
|-------|--------|
| **State-Design** | TypedDict mit mehreren Feldern, Reducer vs. Überschreiben |
| **Node-Regeln** | Signatur, partieller Return, Fehlerbehandlung |
| **Edges** | `add_edge()`, kurze Vorschau auf `add_conditional_edges()` |
| **Kompilierung** | `compile()`, `recursion_limit`, Graph-Visualisierung |
| **Ausführung** | `invoke()`, `stream()`, State nach Ausführung inspizieren |
| **LangSmith** | Trace mit `run_name`, Node-by-Node-Sicht |


In [2]:
#@markdown   <p><font size="4" color='green'>  StateGraph</font> </br></p>



diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    STATE[/"📦 State\n(zentrale Wahrheit)"/]
    NODE_A["Node A\nliest State\nschreibt State"] --> STATE
    STATE --> NODE_B["Node B\nliest State\nschreibt State"]
    STATE --> EDGE{"Edge /\nRouter"}
    EDGE -->|"Pfad 1"| NODE_A
    EDGE -->|"Pfad 2"| NODE_B

    style STATE fill:#FF9800,color:#fff
    style EDGE  fill:#F44336,color:#fff
    style NODE_A fill:#4CAF50,color:#fff
    style NODE_B fill:#2196F3,color:#fff
'''
mermaid(diagram, width=800)


**Das Praxisbeispiel: Qualitäts-Agent**

Es wird ein **zweistufiger Analyse-Agent** gebaut:

- **Node 1 `entwurf_node`**: Erstellt einen ersten Textentwurf
- **Node 2 `korrektorat_node`**: Verbessert den Entwurf

Jeder Node liest aus dem **gemeinsamen State** und schreibt seine Ergebnisse zurück.

In [3]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(BASELINE, temperature=0.7)

In [4]:
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    ST([START])
    N1["<b>Node 1</b>\nentwurf_node()\n▸ liest: anfrage\n▸ schreibt: entwurf"]
    N2["<b>Node 2</b>\nkorrektorat_node()\n▸ liest: entwurf\n▸ schreibt: finale_antwort"]
    FIN([FINISH])
    S[/"<b>State</b>\nanfrage: str\nentwurf: str\nfinale_antwort: str\nschritt: int"/]

    ST --> N1 --> N2 --> FIN
    N1 <-.->|"lesen / schreiben"| S
    N2 <-.->|"lesen / schreiben"| S

    style ST fill:#90EE90
    style FIN fill:#FFB6C1
    style S fill:#E8E8FF,stroke:#9999CC
'''

mermaid(diagram, width=900)

# 2 | StateGraph erstellen
---



**Der State: Einzige Wahrheitsquelle**

Der **State** ist das Herzstück jedes LangGraph-Workflows.  
Alle Nodes lesen daraus und schreiben dorthin – sie kommunizieren **nie direkt**.



**TypedDict: Schnell, typsicher, empfohlen**

| Kriterium | TypedDict | Pydantic BaseModel |
|-----------|----------|--------------------|
| **Empfehlung** | ✅ Für Graph-State | ⚠️ Nur für API-Grenzen |
| **Performance** | Kein Overhead | Langsamer (Validation) |
| **Typen** | Statische Hints | Strikte Laufzeit-Validation |



**Reducer: Wie der State aktualisiert wird**

LangGraph unterscheidet zwei Verhaltensweisen bei State-Updates:

| Typ | Verhalten | Beispiel |
|-----|-----------|----------|
| **Überschreiben** (Standard) | Neuer Wert ersetzt alten | `schritt: 2` → `schritt: 3` |
| **Reducer** (`add_messages`) | Neue Werte werden **angehängt** | `[msg1]` + `[msg2]` → `[msg1, msg2]` |

> **Wichtig:** Für `messages` immer `Annotated[list, add_messages]` verwenden –  
> sonst wird der Gesprächsverlauf bei jedem Node-Return überschrieben!

In [5]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

# ─── State-Definition ────────────────────────────────────────────────────
class QualitätsState(TypedDict):
    """State des Qualitäts-Agenten.

    Alle Felder werden von den Nodes geteilt.
    Jeder Node liest und schreibt nur die Felder, die er benötigt.
    """
    messages:      Annotated[list, add_messages]  # Reducer: immer anhaengen
    anfrage:       str   # Eingabe des Nutzers (unveraendert)
    entwurf:       str   # Node 1 schreibt hier
    finale_antwort: str  # Node 2 schreibt hier
    schritt:       int   # Zaehlt Ausführungsschritte

# ─── Initialer State ─────────────────────────────────────────────────────
start_state: QualitätsState = {
    "messages":       [],
    "anfrage":        "",    # Wird beim Aufruf gesetzt
    "entwurf":        "",
    "finale_antwort": "",
    "schritt":        0,
}
print("State-Schema:")
for feld, wert in start_state.items():
    print(f"  {feld:18s}: {type(wert).__name__}")

State-Schema:
  messages          : list
  anfrage           : str
  entwurf           : str
  finale_antwort    : str
  schritt           : int


In [6]:
from langgraph.graph import StateGraph, START, END

# StateGraph mit dem State-Schema initialisieren
builder = StateGraph(QualitätsState)

print("StateGraph erstellt")
print("Nächste Schritte: Nodes hinzufuegen, Edges verbinden, kompilieren")

StateGraph erstellt
Nächste Schritte: Nodes hinzufuegen, Edges verbinden, kompilieren


# 3 | Nodes definieren
---


**Die 4 Regeln für Node-Funktionen**

Eine Node-Funktion empfängt immer den vollständigen State und gibt ausschließlich ein `dict` zurück — also `def node(state: State) -> dict:`. Dabei müssen nur die veränderten Felder zurückgegeben werden; LangGraph merged automatisch mit dem bestehenden State, sodass unveränderte Felder erhalten bleiben. Treten Fehler auf, werden diese nicht mit `raise` weitergeleitet, sondern als String in ein Fehlerfeld des States geschrieben — das hält den Graphen stabil. Schließlich gilt: Nodes sind pure Funktionen ohne globale Seiteneffekte; der State ist die einzige Kommunikationsebene zwischen Nodes.



**Partieller Return – warum kein vollständiger State?**

LangGraph **merged** automatisch:

```python
# Aktueller State:
# { anfrage: "Hallo", entwurf: "", schritt: 0 }

# Node gibt zurück:
return {"entwurf": "Erster Entwurf...", "schritt": 1}

# Neuer State nach Merge:
# { anfrage: "Hallo", entwurf: "Erster Entwurf...", schritt: 1 }
```

> `anfrage` bleibt erhalten, obwohl der Node es nicht zurückgegeben hat!

In [7]:
from langchain_core.messages import HumanMessage, AIMessage

entwurf_prompt     = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m09_entwurf_prompt.md", mode="T")
korrektorat_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m09_korrektorat_prompt.md", mode="T")

# ─── Node 1: Entwurf erstellen ───────────────────────────────────────────
def entwurf_node(state: TextState) -> dict:
    """Erstellt einen ersten Textentwurf."""
    prompt_value = entwurf_prompt.invoke({"anfrage": state["anfrage"]})
    response = llm.invoke(prompt_value)
    return {"entwurf": response.content}

# ─── Node 2: Korrektorat ─────────────────────────────────────────────────
def korrektorat_node(state: TextState) -> dict:
    """Verbessert Sprache und Struktur des Entwurfs."""
    prompt_value = korrektorat_prompt.invoke({"entwurf": state["entwurf"]})
    response = llm.invoke(prompt_value)
    return {"entwurf": response.content}


NameError: name 'TextState' is not defined

In [ ]:
# ─── Node 2: Korrektorat durchführen ────────────────────────────────────
def korrektorat_node(state: QualitätsState) -> dict:
    """Verbessert den Entwurf aus Node 1.

    Liest: state["entwurf"]
    Schreibt: finale_antwort, schritt, messages
    """
    print(f"  [Node 2] Verbessere Entwurf ({len(state['entwurf'])} Zeichen)")

    prompt_value = korrektorat_prompt.invoke({"entwurf": state["entwurf"]})
    response = llm.invoke(prompt_value.messages)

    return {
        "messages": [*prompt_value.messages, response],
        "finale_antwort": response.content,
        "schritt": state["schritt"] + 1,
    }

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TB
    subgraph Node1["entwurf_node() – Node 1"]
        N1R["Liest: anfrage"] --> N1L["LLM: Erstellt Entwurf"]
        N1L --> N1W["Schreibt: entwurf, schritt"]
    end

    subgraph State["Gemeinsamer State"]
        direction LR
        F1["anfrage: str"]
        F2["entwurf: str"]
        F3["finale_antwort: str"]
        F4["schritt: int"]
        F5["messages: list"]
    end

    subgraph Node2["korrektorat_node() – Node 2"]
        N2R["Liest: entwurf"] --> N2L["LLM: Verbessert Text"]
        N2L --> N2W["Schreibt: finale_antwort, schritt"]
    end

    Node1 <-->|merge| State
    Node2 <-->|merge| State

    style State fill:#E8E8FF,stroke:#9999CC
    style Node1 fill:#E8FFE8
    style Node2 fill:#FFE8E8
'''

mermaid(diagram, width=1050)

# 4 | Edges verbinden
---



**Edge-Typen**

| Typ | Methode | Wann verwenden |
|-----|---------|----------------|
| **Einfache Edge** | `add_edge(from, to)` | Immer dieser Weg |
| **Conditional Edge** | `add_conditional_edges(from, fn)` | Entscheidung zur Laufzeit |

**START und END**

```python
from langgraph.graph import START, END

# START = spezieller Eingangsknoten (kein Node, kein Code)
# END   = spezieller Ausgangsknoten (Workflow beenden)
builder.add_edge(START, "mein_node")   # Einstieg
builder.add_edge("mein_node", END)     # Ausstieg
```

**Conditional Edges – kurze Vorschau (Details folgen im nächsten Modul)**

```python
# Routing-Funktion: gibt den Namen des nächsten Nodes zurück
def routing_fn(state: State) -> str:
    if state["Qualität"] == "schlecht":
        return "nochmal_verbessern"   # Schleife
    return END                         # Fertig

builder.add_conditional_edges(
    "korrektorat",    # Von diesem Node
    routing_fn,       # Diese Funktion entscheidet
)
```

> *Conditional Routing & Tool-Loop* zeigt Conditional Edges und Tool-Loops im Detail.

In [ ]:
# Nodes zum Builder hinzufügen
builder.add_node("entwurf",    entwurf_node)
builder.add_node("korrektorat", korrektorat_node)

# Edges verbinden: START → entwurf → korrektorat → END
builder.add_edge(START,         "entwurf")
builder.add_edge("entwurf",     "korrektorat")
builder.add_edge("korrektorat", END)

print("Nodes registriert:", list(builder.nodes.keys()))
print("Graph-Struktur:")
print("  START → entwurf → korrektorat → END")

# 5 | Graph kompilieren & testen
---



**Kompilierung: Builder → unveränderlicher Graph**

Nach `compile()` kann der Graph **nicht mehr verändert** werden.  
Die Kompilierung prüft den Graphen auf Vollständigkeit und erzeugt den ausführbaren Workflow.

| Option | Beschreibung | Standard |
|--------|-------------|----------|
| `checkpointer=` | Persistenz (MemorySaver/PostgresSaver) | Kein Checkpointing |
| `interrupt_before=` | Nodes bei denen pausiert wird | `[]` |
| `interrupt_after=` | Nodes nach denen pausiert wird | `[]` |

> **Best Practice:** Immer `draw_mermaid_png()` nach `compile()` aufrufen –  
> zeigt den **tatsächlich** kompilierten Graphen.

In [ ]:
# Graph kompilieren
graph = builder.compile()

In [ ]:
# Graph visualisieren
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# Graph ausführen mit invoke()
initial_state: QualitätsState = {
    "messages":       [],
    "anfrage":        "Die Bedeutung von RAG-Systemen in modernen KI-Anwendungen",
    "entwurf":        "",
    "finale_antwort": "",
    "schritt":        0,
}

print("Starte Graphen...\n")
ergebnis = graph.invoke(initial_state)

mprint(f"""
## Ergebnis nach invoke()

**Schritte ausgeführt:** {ergebnis["schritt"]}

**Entwurf (Node 1):**
{ergebnis["entwurf"]}

---

**Finale Antwort (Node 2):**
{ergebnis["finale_antwort"]}

**Nachrichten im State:** {len(ergebnis["messages"])} Eintraege
""")

In [ ]:
# State nach Ausführung inspizieren
print("=== State-Inspektion ===")
for feld, wert in ergebnis.items():
    if feld == "messages":
        print(f"  messages: {len(wert)} Eintraege")
        for i, msg in enumerate(wert):
            typ = type(msg).__name__
            inhalt = msg.content[:60].replace("\n", " ") + "..."
            print(f"    [{i}] {typ}: {inhalt}")
    elif isinstance(wert, str) and len(wert) > 60:
        print(f"  {feld}: {wert[:60]}...")
    else:
        print(f"  {feld}: {repr(wert)}")

In [ ]:
# Graph mit stream() Schritt für Schritt verfolgen
print("=== stream() – Node-by-Node ===\n")

stream_state: QualitätsState = {
    **initial_state,
    "anfrage": "Warum ist Checkpointing in LangGraph wichtig?",
}

for schritt, event in enumerate(graph.stream(stream_state, stream_mode="updates")):
    node_name = list(event.keys())[0]
    node_output = event[node_name]

    print(f"--- Schritt {schritt + 1}: Node '{node_name}' ---")
    for feld, wert in node_output.items():
        if feld == "messages":
            print(f"  messages: +{len(wert)} neue Eintraege")
        elif isinstance(wert, str) and len(wert) > 80:
            print(f"  {feld}: {wert[:80]}...")
        else:
            print(f"  {feld}: {repr(wert)}")
    print()

# 6 | Graph im LangSmith
---



LangGraph-Traces in LangSmith zeigen eine **Node-by-Node-Sicht** des Workflows –
ideal zum Verstehen und Debuggen.

**Was LangSmith für StateGraphen zeigt**

| Ebene | Inhalt | Nutzen |
|-------|--------|--------|
| **Graph-Ebene** | Gesamtlaufzeit, Input/Output | Überblick |
| **Node-Ebene** | Ein Eintrag pro Node | Welcher Node lief wie lang? |
| **LLM-Ebene** | Prompt, Response, Tokens | Prompt-Debugging |
| **State-Diff** | State vor/nach jedem Node | Was hat sich verändert? |

**run_name und Tags**

```python
config = {
    "run_name": "Qualitäts-Agent",
    "tags":     ["m09", "zwei-node-graph"],
}
graph.invoke(state, config=config)
```

In [ ]:
# Ausführung mit LangSmith-Konfiguration
langsmith_config = {
    "run_name": "M09_StateGraph_Basics",
    "tags":     ["m09", "zwei-node-graph", "invoke"],
    "metadata": {"modul": "M09", "version": "1.0"},
}

trace_state: QualitätsState = {
    "messages":       [],
    "anfrage":        "Erklaere den Unterschied zwischen invoke() und stream() in LangGraph",
    "entwurf":        "",
    "finale_antwort": "",
    "schritt":        0,
}

print("Starte Trace...\n")
trace_result = graph.invoke(trace_state, config=langsmith_config)

mprint(f"""
## LangSmith-Trace

**Projekt:** M09-StateGraph-Basics
**Run-Name:** Qualitäts-Agent-Demo
**Schritte:** {trace_result["schritt"]}

**Finale Antwort:**
{trace_result["finale_antwort"][:400]}...

> Trace in LangSmith unter Projekt **M09-StateGraph-Basics** einsehen.
""")

**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M09-StateGraph-Basics", limit=3, show_steps=True)

# 7 | Ausblick: Agent-Loop mit LangGraph
---


Das fruehere Kapitel 6 aus M14 gehoert fachlich hierher: Sobald ein Agent nicht nur eine einzelne RAG-Antwort erzeugt, sondern einen kontrollierbaren Loop mit State, Memory oder HITL braucht, ist LangGraph der passende Ort.

In M09 bleibt dieser Abschnitt ein **Architektur-Ausblick**. Die RAG-spezifische Umsetzung wird später in M14/M17/M20 wieder aufgegriffen.

| Frage | Einfache Agenten-API | LangGraph-Variante |
|---|---|---|
| Wer steuert den Ablauf? | Verborgen in `create_agent()` | Explizite Nodes und Edges |
| Wo liegt der Zustand? | Intern | Im State-Schema |
| Wie wird kontrolliert? | Schwer sichtbar | Conditional Edges, Interrupts, Checkpointer |
| Warum wichtig für Pia? | Schnelle Demo | Nachvollziehbare Research-Assistant-Architektur |

**Merksatz:** M14 zeigt den RAG-Agenten als nutzbare Anwendung; M09 erklärt, warum derselbe Ablauf als Graph kontrollierbarer wird.


# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, es kann aber auch gerne eine andere Herausforderung angegangen werden.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Wenn bei einer Aufgabe eine Blockade entsteht, kann zum Beispiel Gemini in Google Colab verwendet werden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='black' size="5">
Drei-Node-Übersetzungs-Agent
</font></p>

Baue einen StateGraph mit **drei Nodes** für einen Übersetzungs-Workflow:

**State:**
```python
class ÜbersetzungsState(TypedDict):
    messages:          Annotated[list, add_messages]
    originaltext:      str   # Eingabe
    übersetzung:      str   # Node 1 schreibt
    rueckübersetzung: str   # Node 2 schreibt (DE→EN→DE)
    Qualität:         str   # Node 3 bewertet
    schritt:           int
```

**Grundlagen**
1. Definiere einen einfachen State mit den benötigten Feldern.
2. Baue einen linearen Graphen mit mindestens 2 Nodes.
3. Teste den Graph einmal mit `invoke()`.

**✅ Erledigt wenn:** `mein_graph.invoke({...})` gibt einen State mit den erwarteten Feldern zurück — der Selbstcheck läuft ohne `AssertionError`.

In [ ]:
# Grundlagen: StateGraph mit mindestens 2 Nodes
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class MeinState(TypedDict):
    eingabe: str
    ergebnis: str
    qualitaet: str

def node_1(state): return {**state, 'ergebnis': '...'}
def node_2(state): return {**state, 'qualitaet': '...'}

builder = StateGraph(MeinState)
# Nodes + Edges hinzufügen
mein_graph = builder.compile()

**Aufbau**
1. Definiere die 3 Nodes (Übersetzen EN->DE, Rück-Übersetzen DE->EN, Qualität bewerten)
2. Verbinden Sie die Nodes sequentiell: START -> node1 -> node2 -> node3 -> END
3. Kompilieren Sie den Graphen und visualisieren Sie ihn mit `draw_mermaid_png()`
4. Führen Sie einen Test mit einem englischen Satz durch
5. Vergleiche Original und Rück-Übersetzung (wie viel geht verloren?)

**✅ Erledigt wenn:** Die Rück-Übersetzung ist erkennbar ähnlich dem Original; die Visualisierung zeigt alle drei Nodes.

In [ ]:
# Aufbau: 3-Node-Übersetzungs-Agent
class ÜbersetzungsState(TypedDict):
    original: str
    übersetzung: str
    rueckübersetzung: str
    qualitaet: str

# Nodes: übersetze_node → rueckübersetze_node → bewerte_node
# Visualisierung:
# from IPython.display import Image
# Image(mein_graph.get_graph().draw_mermaid_png())

**Vertiefung**
1. Ersetze `invoke()` durch `stream()` und geben Sie den State nach **jedem Node** einzeln aus.
2. Beschreiben Sie in 2-3 Sätzen, was sich im Ausgabe-Verhalten ändert.
3. Aktivieren Sie LangSmith-Tracing und beschreiben Sie, welche Nodes und Kanten im Trace sichtbar sind.
4. Fügen Sie ein viertes Feld `wiederholungsrate: float` in den State ein, das den Anteil identischer Woerter zwischen Original und Rückübersetzung speichert. Berechnen Sie diesen Wert im `Qualitäts_node`.

Speichern Sie die `stream()`-Ausgaben in `stream_ergebnisse` (Liste von State-Dicts).

**✅ Erledigt wenn:** `stream_ergebnisse` enthält den State nach jedem Node; der Unterschied zu `invoke()` ist in einer Zeile dokumentiert.

In [ ]:
# Vertiefung: stream() statt invoke()
stream_ergebnisse = []
for state in mein_graph.stream({'original': 'The sky is blue.'}):
    stream_ergebnisse.append(state)
    print('State nach Node:', state)

# Unterschied dokumentieren:
# stream_beobachtung = 'stream() gibt ... während invoke() ...'

<p><font color='black' size="5">
🔍 Selbstcheck mit `assert`
</font></p>

`assert` prüft eine Bedingung — ist sie `False`, stoppt Python mit einem `AssertionError` und zeigt die Fehlermeldung an:

```python
assert bedingung, "Fehlermeldung"

assert 2 + 2 == 4, "Rechnung falsch"    # ✅ kein Fehler
assert len("Hi") > 5, "Text zu kurz"   # ❌ AssertionError: Text zu kurz
```

**So nutzen Sie den Selbstcheck:**
1. Implementiere den `ÜbersetzungsState` und die 3 Nodes in den Zellen über diesem Block
2. Speichern Sie den kompilierten Graph in **`mein_graph`** (`mein_graph = builder.compile()`)
3. Führe die Zelle unten aus — alle 3 State-Felder (Übersetzung, Rück-Übersetzung, Qualität) werden geprüft


<p><font color='black' size="5">
✅ Selbstcheck
</font></p>

In [ ]:
# ► Speichere deinen kompilierten Graph in 'mein_graph'.

_g = mein_graph  # ← Variablennamen anpassen

assert hasattr(_g, "invoke"), \
    "❌ Graph hat kein invoke() – wurde builder.compile() aufgerufen?"

_r = _g.invoke({
    "messages":          [],
    "originaltext":      "Artificial intelligence changes the world.",
    "übersetzung":      "",
    "rueckübersetzung": "",
    "qualität":         "",
    "schritt":           0,
})

assert "übersetzung" in _r and len(_r["übersetzung"]) > 5, \
    "❌ Kein Übersetzungs-Ergebnis – prüfe Node 1 (übersetzen)"
assert "rueckübersetzung" in _r and len(_r["rueckübersetzung"]) > 5, \
    "❌ Kein Rück-Übersetzungs-Ergebnis – prüfe Node 2 (rück-übersetzen)"
assert "qualität" in _r and len(_r["qualität"]) > 5, \
    "❌ Kein Qualitäts-Ergebnis – prüfe Node 3 (qualität_node)"

print(f"✅ Übersetzungs-Graph durchgelaufen")
print(f"   Original:         {_r['originaltext']}")
print(f"   Übersetzung:      {_r['übersetzung'][:80]}")
print(f"   Rück-Übersetzung: {_r['rueckübersetzung'][:80]}")
print(f"   Qualität:         {_r['qualität'][:80]}")


**Was passiert hier?**

1. `compile(...)` — schließt den Graphen ab und erzeugt das ausführbare Objekt